#  Notebook 04: Model Building — Basic Track (ARIMA)

## Overview
This notebook covers basic time-series forecasting using ARIMA:
1. Prepare city-level daily time-series
2. Test for stationarity (ADF test)
3. Fit auto_arima model
4. Forecast and evaluate (MAE, RMSE, R²)
5. Visualize results with confidence intervals

**Models:** ARIMA with auto_arima  
**Metrics:** MAE, RMSE, R², MAPE  

---

## Step 1: Import Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsmodels.tsa.stattools import adfuller
import pmdarima as pm
import warnings
warnings.filterwarnings('ignore')

sns.set_style("darkgrid")
print(" Libraries imported successfully")

In [ ]:
from pathlib import Path as _P
data_candidates = [_P('data/weather_cleaned.csv'), _P('../data/weather_cleaned.csv'), _P('notebooks/../data/weather_cleaned.csv')]
data_path = None
for p in data_candidates:
    if p.exists():
        data_path = p
        break
if data_path is None:
    raise FileNotFoundError('weather_cleaned.csv not found in data/ or ../data/')
df = pd.read_csv(data_path, parse_dates=['last_updated'])

# Load cleaned data
df = pd.read_csv(data_path, parse_dates=['last_updated'])
df = pd.read_csv(data_path, parse_dates=['last_updated'])

# Select a major city for forecasting (London is a good choice)
city = "London"
city_data = df[df["location_name"] == city].copy()

print(f" Selected city: {city}")
print(f"   Records: {len(city_data):,}")
print(f"   Date range: {city_data['last_updated'].min()} to {city_data['last_updated'].max()}")



## Step 2: Prepare Time-Series Data

In [ ]:
# Create daily time-series (temperature)
city_ts = (
    city_data.set_index("last_updated")["temperature_celsius"]
    .resample("D")  # Daily frequency
    .mean()  # Average temperature per day
    .interpolate()  # Linear interpolation for missing days
)

print(f" Time-series prepared:")
print(f"   Length: {len(city_ts):,} days")
print(f"   Frequency: Daily")
print(f"   Mean: {city_ts.mean():.2f}°C")
print(f"   Std: {city_ts.std():.2f}°C")

# Visualize
fig = px.line(
    x=city_ts.index, y=city_ts.values,
    title=f"Daily Temperature Time-Series: {city}",
    labels={"x": "Date", "y": "Temperature (°C)"},
    template="plotly_dark",
    height=400
)
fig.update_traces(line_color="#FF6B35")
fig.write_html(f"../reports/figures/09_arima_{city.lower()}_timeseries.html")
fig.show()

## Step 3: Stationarity Test (Augmented Dickey-Fuller)

In [ ]:
# ADF test
result = adfuller(city_ts.dropna())

print(f"\n Augmented Dickey-Fuller Test:")
print(f"   Test Statistic: {result[0]:.6f}")
print(f"   p-value: {result[1]:.6f}")
print(f"   Critical values:")
for key, value in result[4].items():
    print(f"      {key}: {value:.3f}")

if result[1] < 0.05:
    print(f"\n Series is STATIONARY (p < 0.05)")
    print(f"   → Can use ARIMA(p,0,q) or I=0")
else:
    print(f"\n Series is NON-STATIONARY (p >= 0.05)")
    print(f"   → May need differencing (d=1 or more)")

## Step 4: Train/Test Split

In [ ]:
# 80/20 split
train_size = int(len(city_ts) * 0.8)
train_ts = city_ts.iloc[:train_size]
test_ts = city_ts.iloc[train_size:]

print(f" Train/Test Split:")
print(f"   Train: {len(train_ts):,} observations ({len(train_ts)/len(city_ts)*100:.1f}%)")
print(f"   Test: {len(test_ts):,} observations ({len(test_ts)/len(city_ts)*100:.1f}%)")

# Visualize split
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=train_ts.index, y=train_ts.values,
    mode='lines', name='Train',
    line=dict(color='steelblue', width=2)
))
fig.add_trace(go.Scatter(
    x=test_ts.index, y=test_ts.values,
    mode='lines', name='Test',
    line=dict(color='green', width=2)
))
fig.update_layout(
    title=f"Train/Test Split: {city}",
    xaxis_title="Date", yaxis_title="Temperature (°C)",
    template="plotly_dark", height=400
)
fig.write_html(f"../reports/figures/10_arima_{city.lower()}_split.html")
fig.show()

## Step 5: Auto-ARIMA Model Fitting

In [ ]:
# Fit auto_arima
print(f" Fitting auto_arima model...")
print(f"   (This may take a minute)\n")

auto_model = pm.auto_arima(
    train_ts,
    seasonal=True,          # Include seasonal component
    m=12,                   # Seasonal period (12 months)
    stepwise=True,          # Use stepwise search (faster)
    suppress_warnings=True,
    error_action="ignore",  # Continue even if error
    max_p=5,                # Max AR order
    max_q=5,                # Max MA order
    max_P=2,                # Max seasonal AR
    max_Q=2,                # Max seasonal MA
    trace=False             # Don't print every iteration
)

print(f" Model fitted!")
print(f"\n{auto_model.summary()}")

## Step 6: Forecast & Evaluate

In [ ]:
# Generate forecast with confidence intervals
forecast_vals, conf_int = auto_model.predict(
    n_periods=len(test_ts),
    return_conf_int=True
)

# Create forecast series
forecast_series = pd.Series(forecast_vals, index=test_ts.index)

# Calculate metrics
mae = mean_absolute_error(test_ts, forecast_vals)
rmse = np.sqrt(mean_squared_error(test_ts, forecast_vals))
r2 = r2_score(test_ts, forecast_vals)
mape = np.mean(np.abs((test_ts - forecast_vals) / test_ts)) * 100

print(f"\n ARIMA Evaluation Metrics:")
print(f"   MAE:  {mae:.3f} °C")
print(f"   RMSE: {rmse:.3f} °C")
print(f"   R²:   {r2:.3f}")
print(f"   MAPE: {mape:.2f}%")

## Step 8: Residual Analysis

In [ ]:
# Calculate residuals
residuals = test_ts.values - forecast_vals

# Create residual plots
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Residual time-series
axes[0, 0].plot(test_ts.index, residuals, color='crimson')
axes[0, 0].axhline(y=0, color='black', linestyle='--', alpha=0.5)
axes[0, 0].set_title('Residuals Over Time', fontweight='bold')
axes[0, 0].set_ylabel('Residual (°C)')
axes[0, 0].grid(True, alpha=0.3)

# Histogram
axes[0, 1].hist(residuals, bins=30, color='steelblue', edgecolor='black')
axes[0, 1].set_title('Distribution of Residuals', fontweight='bold')
axes[0, 1].set_xlabel('Residual (°C)')
axes[0, 1].set_ylabel('Frequency')

# Q-Q plot
from scipy import stats
stats.probplot(residuals, dist="norm", plot=axes[1, 0])
axes[1, 0].set_title('Q-Q Plot', fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# ACF-like: autocorrelation
axes[1, 1].scatter(forecast_vals, residuals, alpha=0.6, color='green')
axes[1, 1].axhline(y=0, color='black', linestyle='--', alpha=0.5)
axes[1, 1].set_title('Residuals vs Fitted Values', fontweight='bold')
axes[1, 1].set_xlabel('Fitted Values (°C)')
axes[1, 1].set_ylabel('Residuals (°C)')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"../reports/figures/12_arima_{city.lower()}_residuals.png", dpi=150, bbox_inches='tight')
plt.show()

print(f" Residual analysis complete")
print(f"   Mean residual: {residuals.mean():.4f} (should be ~0)")
print(f"   Std residual: {residuals.std():.4f}")

## Summary & Save Results

In [ ]:
# Store results
results_arima = {
    "Model": "ARIMA",
    "City": city,
    "MAE": mae,
    "RMSE": rmse,
    "R2": r2,
    "MAPE": mape
}

print("\n" + "="*60)
print("ARIMA MODELING SUMMARY")
print("="*60)
print(f"\n ARIMA Model Performance:")
for key, val in results_arima.items():
    if isinstance(val, float):
        print(f"   {key}: {val:.3f}")
    else:
        print(f"   {key}: {val}")

# Save to CSV for later comparison
results_df = pd.DataFrame([results_arima])
results_df.to_csv("../reports/model_results.csv", index=False)

print(f"\n Notebook 04 Complete!")
print(f"   Results saved to reports/model_results.csv")
print(f"   Ready for Advanced Modeling.")